# 02 预处理 + 三个经典模型

这一份做两件事：
1. 按我们定好的三个决定处理数据：保留重复行但让相同的行落在同一边（决定 1-C）、分层随机切 70/30（决定 2-A）、Amount 稳健缩放 + Time 标准化（决定 3）。
2. 训练逻辑回归、随机森林、SVM 三个模型，填报告第一张指标表。

切好、缩放好的数据会存到 Drive，后面的融合实验直接读，保证所有方案用的是同一份切分。

## 0. 准备：中文字体 + 读数据

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = glob.glob('/usr/share/fonts/**/NotoSansCJK*-Regular.ttc', recursive=True)[0]
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/data-mining'
DRIVE_PATH = f'{DATA_DIR}/creditcard.csv'
MIRROR_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

if os.path.exists(DRIVE_PATH):
    df = pd.read_csv(DRIVE_PATH)
else:
    df = pd.read_csv(MIRROR_URL)
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_csv(DRIVE_PATH, index=False)

assert df.shape == (284807, 31) and (df['Class'] == 1).sum() == 492
print('数据就绪：', df.shape)

## 1. 先看一眼重复的行

"完全相同"指 31 列全部一样：同一秒（Time 只精确到秒）、同样金额、28 个脱敏特征也一模一样。
看什么：重复的欺诈交易金额是不是偏小。有一种真实存在的欺诈叫"盗刷测卡"：拿到别人的卡号后，先用小金额反复刷几次，确认这张卡能用。

In [ ]:
dup_all = df[df.duplicated(keep=False)]
print('涉及重复的行数（含原件）：', len(dup_all))
print('其中欺诈行数：', dup_all['Class'].sum())
print()
print('重复行的金额，按类别看：')
print(dup_all.groupby('Class')['Amount'].describe().round(2))

## 2. 切分训练集和测试集（决定 1-C + 决定 2-A）

做法：
1. 给每一种"行的内容"一个编号，完全相同的行编号相同。
2. 只对"不同的内容"做分层随机切分（70% 训练 / 30% 测试，保持两边欺诈比例一致）。
3. 每一行跟着自己的编号走，所以完全相同的行一定落在同一边，不会出现"训练集见过原题"。

`random_state=42` 固定了随机种子，每次运行切出来都一样。

In [ ]:
from sklearn.model_selection import train_test_split

row_id = pd.util.hash_pandas_object(df, index=False)
unique_rows = df.assign(row_id=row_id.values).drop_duplicates('row_id')

train_ids, test_ids = train_test_split(
    unique_rows['row_id'], test_size=0.3,
    stratify=unique_rows['Class'], random_state=42,
)
is_train = row_id.isin(set(train_ids)).values

train_df = df[is_train].reset_index(drop=True)
test_df = df[~is_train].reset_index(drop=True)

print(f'训练集：{len(train_df)} 行，其中欺诈 {train_df.Class.sum()} 条，占 {train_df.Class.mean():.4%}')
print(f'测试集：{len(test_df)} 行，其中欺诈 {test_df.Class.sum()} 条，占 {test_df.Class.mean():.4%}')

overlap = set(row_id[is_train]) & set(row_id[~is_train])
print('同时出现在训练集和测试集的相同行：', len(overlap), '（应该是 0）')

## 3. 缩放（决定 3）

- Amount 用稳健缩放（减中位数、除以四分位距），不受极端大金额影响。
- Time 用标准化（减均值、除以标准差）。
- V1~V28 已经在 0 附近，不动。

关键：缩放规则只用训练集来算（`fit`），再原样套用到测试集（`transform`），测试集的信息不能参与。

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler

amount_scaler = RobustScaler().fit(train_df[['Amount']])
time_scaler = StandardScaler().fit(train_df[['Time']])

for part in (train_df, test_df):
    part['Amount'] = amount_scaler.transform(part[['Amount']])
    part['Time'] = time_scaler.transform(part[['Time']])

feature_cols = [c for c in df.columns if c != 'Class']
X_train, y_train = train_df[feature_cols], train_df['Class']
X_test, y_test = test_df[feature_cols], test_df['Class']
print(X_train[['Time', 'Amount']].describe().round(2))

存一份到 Drive，后面的笔记本直接读，不用重新切。

In [ ]:
train_df.to_csv(f'{DATA_DIR}/train_scaled.csv', index=False)
test_df.to_csv(f'{DATA_DIR}/test_scaled.csv', index=False)
print('已保存到', DATA_DIR)

## 4. 评估函数

每个模型都算同一组指标：
- Accuracy、Precision、Recall、F1：前三个你已经熟悉，都只看"欺诈"这一类。
- ROC-AUC：模型把"欺诈排在正常前面"的整体能力，0.5 是瞎猜，1 是完美。在极度不平衡时容易显得很高。
- AUPRC：Precision-Recall 曲线下的面积，只关心欺诈这一类，更适合这份数据。瞎猜的水平约等于欺诈占比，也就是 0.0017 左右。
- 训练耗时：秒。

AUPRC 和 ROC-AUC 用的是模型给出的"欺诈分数"，不是 0/1 的预测，所以不受阈值影响。

In [ ]:
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

results = []

def evaluate(name, y_pred, y_score, train_seconds):
    row = {
        '模型': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_score),
        'AUPRC': average_precision_score(y_test, y_score),
        '训练耗时(秒)': train_seconds,
    }
    results.append(row)
    return pd.Series({k: v for k, v in row.items() if k != '模型'}, name=name).round(4)

## 5. 逻辑回归（基线）

`class_weight='balanced'`：和比特币项目里一样，让模型别偷懒全猜"正常"。

In [ ]:
from sklearn.linear_model import LogisticRegression

start = time.time()
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)
t = time.time() - start

evaluate('逻辑回归', lr.predict(X_test), lr.predict_proba(X_test)[:, 1], t)

## 6. 随机森林

很多棵决策树投票。树模型不需要缩放，但用缩放后的数据也不影响结果。大约要跑 1~3 分钟。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
t = time.time() - start

evaluate('随机森林', rf.predict(X_test), rf.predict_proba(X_test)[:, 1], t)

## 7. SVM

问题：训练集接近 20 万行，RBF 核 SVM 的训练时间大约随样本数的平方增长，全量训练在 Colab 上可能要几个小时。

这里的做法：训练时保留全部欺诈样本，正常样本随机抽 `N_NORMAL` 条（先设 20000）。测试仍然用完整测试集，所以成绩和另外两个模型可比。
这是一个"速度 vs 信息量"的取舍，报告里要写。想试别的数量，改 `N_NORMAL` 重跑即可。

SVM 的"欺诈分数"用 `decision_function`，即样本到分界线的有符号距离。

In [ ]:
from sklearn.svm import SVC

N_NORMAL = 20000

svm_train = pd.concat([
    train_df[train_df.Class == 1],
    train_df[train_df.Class == 0].sample(N_NORMAL, random_state=42),
])

start = time.time()
svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm.fit(svm_train[feature_cols], svm_train['Class'])
t = time.time() - start

evaluate(f'SVM（正常样本抽 {N_NORMAL}）', svm.predict(X_test), svm.decision_function(X_test), t)

## 8. 三个模型汇总（报告第一张表）

In [ ]:
table = pd.DataFrame(results).set_index('模型').round(4)
table.to_csv(f'{DATA_DIR}/results_classic.csv')
table

## 9. 类别不平衡处理对比（报告"类别不平衡处理"一栏）

用最快的逻辑回归，比较三种做法：什么都不做、类别权重、SMOTE 过采样。
SMOTE 只对训练集做，测试集保持原样。

In [ ]:
!pip -q install imbalanced-learn
from imblearn.over_sampling import SMOTE

compare = []
variants = {
    '不处理': (X_train, y_train, None),
    '类别权重': (X_train, y_train, 'balanced'),
}
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
variants['SMOTE'] = (X_sm, y_sm, None)

for name, (Xt, yt, cw) in variants.items():
    model = LogisticRegression(class_weight=cw, max_iter=1000).fit(Xt, yt)
    pred, score = model.predict(X_test), model.predict_proba(X_test)[:, 1]
    compare.append({
        '做法': name,
        '训练集欺诈占比': yt.mean(),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'AUPRC': average_precision_score(y_test, score),
    })

pd.DataFrame(compare).set_index('做法').round(4)

## 看完之后记下来（用你自己的话）

1. 重复行里的欺诈金额是不是偏小？像不像"盗刷测卡"？
2. 三个模型里，谁的 Recall 最高？谁的 Precision 最高？AUPRC 谁最高？
3. 哪个模型 Accuracy 最高？它是不是也是 AUPRC 最高的？这说明了什么？
4. SVM 只用了一部分正常样本，训练耗时和成绩跟另外两个比怎么样？
5. 不处理、类别权重、SMOTE 三种做法，Precision 和 Recall 是怎么此消彼长的？